In [0]:
%pip install databricks-labs-dqx==0.13.0

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("config_catalog_name", "", "config_catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")
dbutils.widgets.text("target_schema_name", "", "target_schema_name")
dbutils.widgets.text("table_name", "", "table_name")

config_catalog_name = dbutils.widgets.get("config_catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")
target_schema_name = dbutils.widgets.get("target_schema_name")
full_table_name = dbutils.widgets.get("table_name")

print("config_catalog_name: ", config_catalog_name)
print("config_schema_name: ", config_schema_name)
print("target_schema_name: ", target_schema_name)
print("table_name: ", full_table_name)

In [0]:
source_catalog_name = full_table_name.split('.')[0]
table_name = full_table_name.split('.')[2]
print("source_catalog_name: ", source_catalog_name)
print("table_name: ", table_name)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {source_catalog_name}.{target_schema_name}")

In [0]:
query = f"""
SELECT
    m.table_name AS table_name,
    r.rule_name,
    m.column_name AS column,
    r.rule_function AS function,
    m.criticality AS criticality,
    m.arguments AS arguments,
    r.rule_type,
    m.is_active
FROM {config_catalog_name}.{config_schema_name}.dqx_rule_mappings m
JOIN {config_catalog_name}.{config_schema_name}.dqx_rule_definitions r ON m.rule_id = r.rule_id
WHERE lower(m.table_name) = lower('{full_table_name}')
"""

dqx_mapped_df = spark.sql(query).filter("is_active=true")
display(dqx_mapped_df)

if dqx_mapped_df.isEmpty():
    raise Exception("No active rules found for the table")

## Apply DQX Checks

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import pyspark.sql.functions as F
import datetime

from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule,DQDatasetRule
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.config import TableChecksStorageConfig
from databricks.labs.dqx.config import InputConfig, OutputConfig
from databricks.labs.dqx.metrics_observer import DQMetricsObserver
from databricks.labs.dqx.io import read_input_data, save_dataframe_as_table

In [0]:
# 4. Initialize Engine and Run
observer = DQMetricsObserver(name="my_observation")
ws = WorkspaceClient()
engine = DQEngine(ws, observer=observer)

## config - check by metadata

In [0]:
checks_config = []
for row in dqx_mapped_df.collect():
    import json
    args = {}
    if row['arguments']:
        for k, v in row['arguments'].items():
            try:
                args[k] = json.loads(v)
            except Exception:
                args[k] = v
    check_dict = {
        "criticality": row['criticality'],
        "check": {
            "function": row['function'],
            "arguments": {k: (True if str(v).upper() == 'TRUE' else False if str(v).upper() == 'FALSE' else v) for k, v in args.items()}
        }
    }
    checks_config.append(check_dict)

In [0]:
for i in (checks_config):
    print(i)

In [0]:
# target_date = datetime.date(2026, 5, 13)
target_date = str(datetime.date.today())
print("target_date = ",target_date)

In [0]:
input_config = InputConfig(f"{full_table_name}")

output_config = OutputConfig(f"{source_catalog_name}.{target_schema_name}.{table_name}_output", mode="overwrite", options={"mergeSchema": "true"})

quarantine_config = OutputConfig(f"{source_catalog_name}.{target_schema_name}.{table_name}_quarantine", mode="overwrite", options={"mergeSchema": "true", "partitionBy":"dqx_run_date", "replaceWhere": f"dqx_run_date = '{target_date}'" })

metrics_config = OutputConfig(f"{config_catalog_name}.{config_schema_name}.summary_metrics", mode="append", options={"mergeSchema": "true" ,"partitionBy": "input_location"})

In [0]:
valid_df, quarantine_df, batch_observation = engine.apply_checks_by_metadata_and_split(
    spark.table(f"{full_table_name}"),
    checks=checks_config
)
print("valid_df count: ", valid_df.count())
print("quarantine_df count: ", quarantine_df.count())

In [0]:
# quarantine_df = quarantine_df.withColumn("dqx_run_date", F.lit('2026-05-13').cast('date'))
quarantine_df = quarantine_df.withColumn("dqx_run_date", F.current_date())

### Save Tables

In [0]:
save_dataframe_as_table(valid_df, output_config)

In [0]:
save_dataframe_as_table(quarantine_df , quarantine_config)

In [0]:
engine.save_summary_metrics(
    observed_metrics=batch_observation.get,
    metrics_config=metrics_config,
    input_config=input_config,
    output_config=output_config,
    quarantine_config=quarantine_config,
    checks_location=None,
)

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {source_catalog_name}.{target_schema_name}.{table_name}_quarantine_vw AS
SELECT 
    q.*,
    q._error_exploded.function AS dqx_function,
    q._error_exploded.columns AS column_list,
    rd.rule_id,
    rd.rule_name,
    rd.rule_dimension,
    regexp_extract(q._error_exploded.message, "Value '([^']+)' in Column", 1) AS key_value
FROM (
    SELECT *, EXPLODE(_errors) AS _error_exploded
    FROM {source_catalog_name}.{target_schema_name}.{table_name}_quarantine
) q
LEFT JOIN {config_catalog_name}.{config_schema_name}.dqx_rule_definitions rd
    ON q._error_exploded.function = rd.rule_function
""")